In [19]:
import pandas as pd
import numpy as np

# input path to where the raw canvas grades csv file is
input_path = '/Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/NEUROSCI380L/2026-04-27T1128_Grades-NEUROSCI_380L.01L.Sp26.csv'

# output path to save the modified csv file to
output_path = '/Users/jinjiang-macair/Library/CloudStorage/OneDrive-DukeUniversity/Duke/NEUROSCI380L/Final_Modified_Grades_NEUROSCI_380L_01L_Sp26_with_Averages.csv'

In [20]:
# Load the raw canvas grades csv file
df = pd.read_csv(input_path)

# Convert all score columns to float, handling non-numeric entries
for col in df.columns[4:]:  # Adjust this as necessary
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Extract points possible from the first row after the headers and convert them to float
points_possible = df.iloc[0, 4:].astype(float)  # Adjust index if your data starts elsewhere

# Convert scores to percentages, starting from the second row (index 1)
df.iloc[1:, 4:] = df.iloc[1:, 4:].div(points_possible)

# Drop the 'Points Possible' row if it's no longer needed
df = df.drop(index=0)

# USERS SHOULD UPDATE THE EXCLUDE STRINGS AS NEEDED. 
# Define the categories and their weights, allowing for partial matches in the column names.
# Exclude strings will exclude assignments with these strings in their name from being included in the grade average for that category.
# In particular, if you make an assignment for the self-directed learning seminar that has 'IRA', 'TRA', or 'Clinical Case Set' in the name, make sure to exclude it from being part of those categories.
# For example, in 2026, I had a clinical case set assignment for the self-directed learning seminar that I do not want graded, so I exclude it from the clinical case set category.

categories = {
    'TRA': {'weight': 0.20, 'exclude': ['Final Exam: TRA', 'Exam', 'myths and mysteries']},
    'IRA': {'weight': 0.35, 'exclude': ['Final Exam: IRA', 'Exam', 'myths and mysteries']},
    'Clinical case set': {'weight': 0.20, 'exclude': ['Clinical case set (self-directed learning seminar) (322896)']},
    'Self-Directed Learning Seminar Written Critique Upload': {'weight': 0.05, 'exclude': None},
    'TRA: Final Exam': {'weight': 0.10, 'exclude': ['TRA: Final exam 2025']},
    'IRA: Final Exam': {'weight': 0.10, 'exclude': None}
}

# Initialize a column for the final average
df['Final Average'] = 0.0

# To store used columns for each category
used_columns = {}

# --- Pre-step: find IRA + TRA columns and mask the 2 lowest combined scores per student ---
def find_columns(category, info):
    return [col for col in df.columns if category.lower() in col.lower() and
            all(excl.lower() not in col.lower() for excl in (info['exclude'] if info['exclude'] else [])) and
            'score' not in col.lower() and 'grade' not in col.lower()]

ira_cols = find_columns('IRA', categories['IRA'])
tra_cols = find_columns('TRA', categories['TRA'])
combined_cols = ira_cols + tra_cols

# For each student, find indices of the 2 lowest scores across IRA+TRA and set them to NaN
combined = df[combined_cols].to_numpy(dtype=float)
# argsort puts NaNs last, so the 2 lowest *actual* scores come first
lowest_idx = np.argsort(combined, axis=1)[:, :2]
rows = np.arange(combined.shape[0])[:, None]
combined[rows, lowest_idx] = np.nan
df[combined_cols] = combined

# --- Main loop: now IRA and TRA can be averaged normally ---
for category, info in categories.items():
    columns = find_columns(category, info)
    if not columns:
        print(f'No columns found for category: {category}')
        continue

    df[category + ' Average'] = df[columns].mean(axis=1)  # NaN-aware via skipna=True default
    df['Final Average'] += df[category + ' Average'] * info['weight']
    used_columns[category] = columns
    print(category, used_columns[category])

# Save the modified DataFrame to a new CSV file
df.to_csv(output_path, index=False)

# Print only the columns that exist in the DataFrame
columns_to_print = ['Student', 'Final Average'] + [col for col in df.columns if 'Average' in col and col != 'Final Average']
print(f"DataFrame saved to {output_path}")
print(df[columns_to_print])

# Print used columns for each specific category
for category in ['TRA', 'IRA', 'Clinical case set']:
    print(f"Columns used for {category} calculations: {used_columns[category]}")


TRA ['TRA: Blood supply to the brain and spinal cord (322899)', 'TRA: Surface anatomy of the forebrain (322910)', 'TRA: Internal anatomy of the forebrain  (322904)', 'TRA: Surface anatomy of the brainstem and spinal cord  (322909)', 'TRA: Hypothalamus and visceral motor system (322902)', 'TRA: Internal anatomy of the spinal cord and brainstem (322905)', 'TRA: Somatic sensory pathways  (322908)', 'TRA: Visual pathways  (322912)', 'TRA: Upper and Lower Motor Neurons  (322911)', 'TRA: Basal ganglia  (322898)', 'TRA: Cerebellum  (322900)', 'TRA: Limbic system (322906)']
IRA ['IRA: Internal anatomy of the forebrain (322863)', 'IRA: Internal anatomy of the forebrain_makeup (340406)', 'IRA: Blood supply to the brain and spinal cord (322850)', 'IRA: Blood supply to the brain and spinal cord_makeup (340405)', 'IRA: Surface anatomy of the forebrain (322886)', 'IRA: Surface anatomy of the forebrain makeup (341886)', 'IRA: Surface anatomy of the forebrain___makeup (337346)', 'IRA: Surface anatomy 

/var/folders/f0/1x9ky0yd3nj4p68sphk83ngr0000gn/T/ipykernel_38339/1363529901.py:33: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['Final Average'] = 0.0
/var/folders/f0/1x9ky0yd3nj4p68sphk83ngr0000gn/T/ipykernel_38339/1363529901.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[category + ' Average'] = df[columns].mean(axis=1)  # NaN-aware via skipna=True default
/var/folders/f0/1x9ky0yd3nj4p68sphk83ngr0000gn/T/ipykernel_38339/1363529901.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the resu